In [56]:
import pandas as pd;

df_aero = pd.read_csv('../Data/AeroSCOPE_global_aviation_traffic_dataset_16_11.csv',
                       keep_default_na=False, na_values=[''])
df_owid = pd.read_csv('../Data/owid-co2-data.csv')

print(df_aero.shape)
print(df_owid.shape)

print(df_aero.columns)
print(df_owid.columns)






(293838, 26)
(50411, 79)
Index(['Unnamed: 0', 'airline_iata', 'acft_icao', 'acft_class',
       'seymour_proxy', 'source', 'seats', 'n_flights', 'iata_departure',
       'iata_arrival', 'departure_lon', 'departure_lat', 'departure_country',
       'departure_continent', 'arrival_lon', 'arrival_lat', 'arrival_country',
       'arrival_continent', 'seats_no_est_scaling', 'distance_km', 'ask',
       'rpk', 'fuel_burn_seymour', 'fuel_burn', 'co2', 'domestic'],
      dtype='str')
Index(['country', 'year', 'iso_code', 'population', 'gdp', 'cement_co2',
       'cement_co2_per_capita', 'co2', 'co2_growth_abs', 'co2_growth_prct',
       'co2_including_luc', 'co2_including_luc_growth_abs',
       'co2_including_luc_growth_prct', 'co2_including_luc_per_capita',
       'co2_including_luc_per_gdp', 'co2_including_luc_per_unit_energy',
       'co2_per_capita', 'co2_per_gdp', 'co2_per_unit_energy', 'coal_co2',
       'coal_co2_per_capita', 'consumption_co2', 'consumption_co2_per_capita',
       'con

In [57]:
print(df_aero.head())
print(df_owid.head())

   Unnamed: 0 airline_iata acft_icao acft_class seymour_proxy source  seats  \
0           0           5V      PC12         TP          PC12    BTS    4.5   
1           1           5V      C208         TP          C208    BTS    4.5   
2           2           5V      PC12         TP          PC12    BTS    4.5   
3           3           5V      C208         TP          C208    BTS    4.5   
4           4           5V      PC12         TP          PC12    BTS    9.0   

   n_flights iata_departure iata_arrival  ...  arrival_country  \
0        0.5            05A          AET  ...               US   
1        0.5            05A          AKP  ...               US   
2        0.5            05A          AKP  ...               US   
3        0.5            05A          ARC  ...               US   
4        1.0            05A          CXF  ...               US   

   arrival_continent seats_no_est_scaling distance_km          ask  \
0                 NA                  4.5  223.942934  100

In [58]:
# recherche du nom de l'A320ceo et A320neo
df_aero[df_aero['acft_icao'].str.contains('A320|A20N')]['acft_icao'].unique()

<StringArray>
['A320', 'A20N']
Length: 2, dtype: str

In [59]:
df_a320 = df_aero[df_aero['acft_icao'].str.contains('A320|A20N')]

df_a320.shape

(37046, 26)

In [60]:
df_a320.isnull().sum()

Unnamed: 0              0
airline_iata            0
acft_icao               0
acft_class              0
seymour_proxy           0
source                  0
seats                   0
n_flights               0
iata_departure          0
iata_arrival            0
departure_lon           0
departure_lat           0
departure_country       0
departure_continent     0
arrival_lon             0
arrival_lat             0
arrival_country         0
arrival_continent       0
seats_no_est_scaling    0
distance_km             0
ask                     0
rpk                     0
fuel_burn_seymour       0
fuel_burn               0
co2                     0
domestic                0
dtype: int64

In [61]:
df_a320.dtypes

Unnamed: 0                int64
airline_iata                str
acft_icao                   str
acft_class                  str
seymour_proxy               str
source                      str
seats                   float64
n_flights               float64
iata_departure              str
iata_arrival                str
departure_lon           float64
departure_lat           float64
departure_country           str
departure_continent         str
arrival_lon             float64
arrival_lat             float64
arrival_country             str
arrival_continent           str
seats_no_est_scaling    float64
distance_km             float64
ask                     float64
rpk                     float64
fuel_burn_seymour       float64
fuel_burn               float64
co2                     float64
domestic                  int64
dtype: object

In [62]:
df_a320[['co2', 'distance_km', 'seats', 'fuel_burn']].describe()

,co2,distance_km,seats,fuel_burn
count,3.704600e+04,37046.000000,37046.000000,3.704600e+04
mean,1.319908e+06,1534.733591,14241.927628,4.176923e+05
std,2.748222e+06,980.184455,31984.937998,8.696905e+05
min,0.000000e+00,0.000000,0.000000,0.000000e+00
25%,3.310374e+04,804.580259,498.187500,1.047587e+04
50%,3.062777e+05,1409.747206,3238.218750,9.692331e+04
75%,1.329003e+06,2109.709557,13700.156250,4.205704e+05
max,5.582050e+07,15872.770078,919986.250000,1.766471e+07


In [63]:
df_a320 = df_a320[(df_a320['co2'] > 0) & (df_a320['distance_km'] > 0)]
df_a320.shape


(36496, 26)

In [71]:
df_a320['type_appareil'] = df_a320['acft_icao'].map({'A320': 'ceo', 'A20N': 'neo'})

df_a320['type_appareil'].value_counts()



type_appareil
ceo    30778
neo     5718
Name: count, dtype: int64

In [65]:
part_a320 = df_a320.shape[0] / df_aero.shape[0] * 100
print(f" {part_a320:.1f}")

 12.4


## Insight #1
Les A320 (ceo + neo) représentent **12.4% des routes mondiales** dans le dataset AeroSCOPE.

In [66]:
df_a320['acft_icao'].value_counts()

acft_icao
A320    30778
A20N     5718
Name: count, dtype: int64

In [67]:
counts = df_a320['acft_icao'].value_counts()
part_neo = counts['A20N'] / counts.sum() * 100
print(f"Le A320neo représente {part_neo:.1f}% de la flotte A320")

Le A320neo représente 15.7% de la flotte A320


##  Insight #2
Sur les 36 496 routes opérées par des A320, seulement **15.7% sont en A320neo**.
84.3% de la flotte A320 est encore en ceo ce qui représente un **énorme potentiel de réduction CO₂** si la transition s'accélère.

In [73]:
df_a320_clean = df_a320[['type_appareil','airline_iata','iata_departure','iata_arrival','departure_country','arrival_country','departure_continent','arrival_continent','distance_km','co2','seats','n_flights']]

In [76]:
df_a320_clean.shape
df_a320_clean.head()

,type_appareil,airline_iata,iata_departure,iata_arrival,departure_country,arrival_country,departure_continent,arrival_continent,distance_km,co2,seats,n_flights
145,ceo,G4,ABE,AVL,US,US,NA,NA,850.005218,10955.546533,186.0,1.0
156,ceo,G4,ABE,BNA,US,US,NA,NA,1101.575025,863672.536078,11799.0,66.0
189,ceo,UA,ABE,DEN,US,US,NA,NA,2476.779362,12825.542159,75.0,0.5
192,ceo,DL,ABE,DTW,US,US,NA,NA,683.246599,4786.014829,78.5,0.5
202,ceo,DL,ABE,EWR,US,US,NA,NA,107.819675,2486.742458,78.5,0.5


In [77]:
df_a320_clean.to_csv('../Data/a320_clean.csv', index=False)